# Cross-validation

With the code developed so far, it is possible to train an ANN and provide an estimate of the results it would offer in its real execution (with unseen patterns, represented by a test set). However, in this last aspect there are two factors to consider, as a consequence of the non-deterministic nature of the process we are following:

- The partitioning of the set of patterns into training/test is random (hold out), and is therefore overly dependent on good or bad luck in choosing training and test patterns.
- ANN training is not deterministic, as the initialisation of the weights is random. As before, it is too dependent on good or bad luck to start the training at a good or bad starting point.

For these two reasons, the test result of a single training is not significant when assessing the goodness of fit of the model in the presence of unseen patterns. To solve this problem, the experiment is repeated several times and the results are averaged. This can be implemented in a simple way by means of a loop; however, it is necessary to do this in an orderly way as there are two different sources of randomness.

Firstly, to minimise the randomness due to the partitioning of the data set, it is necessary to have a method that ensures that each data is used for training at least once, and for testing at least once. The most commonly used method is cross-validation. In this method, the data set is split into k disjoint subsets and k experiments are performed. In the k-th experiment, the k subset is separated for testing, and the remaining k-1 substes are used for training, performing a k-fold cross-validation. A common value is k=10, which gives a 10-fold cross-validation. Finally, the test value corresponding to the appropriate metric will be the average value of the values of the k experiments.

A widely used variant of cross-validation is stratified cross-validation. In this case, each subset is created in such a way as to keep the proportion of patterns of each class the same (or similar) as in the original dataset. This is particularly used when the data set is imbalanced.

It is usual to save not only the mean, but also the k values, in order to subsequently perform a paired hypothesis test with another model. To do this, it is necessary that both models have been trained using the same training and test sets.

This way of evaluating the model is often considered to be slightly pessimistic, i.e. the results obtained in tests are slightly worse than those that would be obtained from real training with all available data. In a hold out experiment, as mentioned above, several data are separated for testing. This means that the model is trained with less data than is available, and that by chance the data separated for testing can be of great importance (especially if there is little data). For this reason, when training with less data and possibly no "important" data, hold out is considered a pessimistic assessment. In the same way, cross-validation also separates data for testing, so it does not train on all available data, and is therefore also pessimistic. However, it is guaranteed that all data are used at least once in training and once in testing, thus trying to minimise the impact of chance in separating data, so it is considered only a slightly pessimistic evaluation.

Doing this is as simple as splitting the data set and performing a loop with k iterations in which at the k-th iteration a model is trained and evaluated with the corresponding sets. However, if the model is not deterministic, the result obtained at the k-th iteration will not be meaningful, since it is again dependent on chance. In this case, what needs to be done is a second nested loop within iteration k in which the model is repeatedly trained, and finally an average of the results is made to finally output the result of iteration k. The number of trainings must be high for the average results to be really significant, at least 50 trainings.

### Question 5.1

> ❓ If this second loop is performed with a deterministic model, what will be the standard deviation of the test results obtained? Is there a difference between performing this second loop and averaging the results, or doing a single training?

`If the model is truly deterministic (same data split and same initialization), repeated trainings give identical test scores so the standard deviation is zero.
Averaging those identical results is therefore identical to reporting a single training result, no difference in value.`

In this way, it is possible to evaluate a model together with its hyperparameters in solving a problem. A very common situation is to compare several models (or the same model with different hyperparameters), for which this scheme has to be applied with an important caveat: the sets used in the cross-validation must be the same for each model. Since the distribution of patterns in different sets is random, having the same subsets in different runs is achieved by setting the random seed at the beginning of the program to be executed. Setting the random seed not only allows the same subsets to be generated, but is also important in order to be able to repeat the results in different runs.

It is also important to bear in mind that this methodology allows estimating the real performance of a model (although slightly pessimistic). The final model that would be used in production would be the result of training it with all the available patterns, since, as seen in the theory class, and very generally speaking, the more patterns you train with, the better the model will be.

Another commonly requested task is to obtain a **confusion matrix on the test set** as a result of performing **cross-validation**.

In general, this is **not directly possible**, because multiple neural networks are trained for each fold.  
If only a single model were trained per fold (as will be the case in the next exercise), it would be possible to compute a confusion matrix for each test set and then sum all of them to obtain a **global test confusion matrix**.

However, since multiple models are trained in each fold here, we use an alternative approach:

- For each fold, a **confusion matrix is computed for every execution**.
- These matrices are then **averaged within each fold**.
- Finally, the averaged matrices from each fold are **summed** to produce a **global confusion matrix**.

---

### ⚠️ Important considerations

- This global confusion matrix **does not represent the result of any single model**.  
  Therefore, calling it a "confusion matrix" can be misleading — but it is still useful to analyze how instances are distributed in the test sets.

- For the same reason, the **metrics derived from this matrix** will generally **not match** the metric values returned by the function (which are averages of the metric scores for each fold).  
  These matrices are meant to provide **a visual and aggregated summary**, while the true evaluation metrics are the ones computed per fold.

- As a result of averaging the confusion matrices, the **final confusion matrix will contain real numbers**, not integers.  
  While this is technically not correct, it is acceptable to include it in a report — as long as the nature of the matrix is clearly explained.

In this assignment, you are asked to:

1. Develop a function called `crossvalidation` that receives a value `N` (equal to the number of patterns), and a value `k` (number of subsets into which the dataset is to be split), and returns a vector of length N, where each element indicates in which subset that pattern should be included.

    To perform this function, one possibility is to perform the following steps:
    
    - Create a vector with k sorted elements, from 1 to k.
    - Create a new vector with repetitions of the previous vector until its length is greater than or equal to N. The functions `repeat` and `ceil` can be used for this purpose.
    - Take the first N values of this vector.
    - Shuffle this vector (using the function `shuffle!` and return it. To use this function, the module `Random` should be loaded.
    
    This function should return a vector of **integer values**, with a length equal to **N**. No loop should be used in this function.

In [141]:
using Random
using Random:seed!

function crossvalidation(N::Int64, k::Int64)
    #TODO
    folds = collect(1:k) # Vector with the k folds
    limit = Int64(ceil(N/k))
    indices = repeat(folds, limit)
    nIndices =  indices[1:N] # Select the first N indexes
    shuffle!(nIndices) # Shuffle indexes
    return nIndices
end

crossvalidation (generic function with 4 methods)

In [142]:
crossvalidation(20,8)

20-element Vector{Int64}:
 3
 8
 4
 3
 5
 5
 6
 2
 3
 2
 4
 7
 6
 1
 8
 4
 1
 2
 7
 1

2. Develop a new function called `crossvalidation` that receives:

- A vector `targets::AbstractArray{Bool,1}` containing the desired outputs (ground truth labels),
- An integer `k`, which is the number of folds for the partitioning.

The function must return a vector of **integer values** of length **N** (where `N` is the number of elements in `targets`), indicating to which subset (fold) each instance belongs.

The partitioning must be **stratified**, meaning that **the distribution of positive and negative instances is preserved across folds**. 
Follow these steps:

    1. Create a vector of indices with as many elements as there are rows in `targets`.
    2. Call the previously developed `crossvalidation` function, passing the **number of positive instances** and the value of `k`.
    3. Assign the result of the previous step to the positions of the index vector that correspond to **positive instances**.
  ### Question 5.2
  > ❓ Could you combine steps 2 and 3 into a single line? (This is not required, but worth considering.)

`Yes, steps 2 and 3 can be combined into a single line by directly assigning the fold numbers to the positive instances. For example, you could write something like:` 

`folds[pos_idx] = crossvalidation(sum(targets), k)`

`This performs both the function call and the assignment in one step, making the code shorter but keeping the same functionality.`

    4. Repeat a similar operation for **negative instances**.
    5. Return the index vector.

Loops are **not allowed** in the implementation of this function. The function must return a **vector of integers** of length **N**.

In [143]:
function crossvalidation(targets::AbstractArray{Bool,1}, k::Int64)
    #TODO
    targets_size = size(targets,1)

    # 1. Create a vector of indices with as many elements as there are rows in `targets`.
    indices = Vector{Int64}(zeros(targets_size))
    
    # 2. Call the previously developed `crossvalidation` function, passing the **number of positive instances** and the value of `k`.
    cv_pos = crossvalidation(sum(targets), k)
    
    # 3. Assign the result of the previous step to the positions of the index vector that correspond to **positive instances**.
    indices[targets] = cv_pos
    
    # 4. Repeat a similar operation for **negative instances**.
    cv_neg = crossvalidation((targets_size-sum(targets)), k) 
    indices[.!targets] = cv_neg 

    # 5. Return the index vector.
    return indices
end

crossvalidation (generic function with 4 methods)

In [144]:
v = Vector{Bool}(undef, 20)
print(v)
crossvalidation(v, 8)

Bool[0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0]

20-element Vector{Int64}:
 4
 8
 1
 5
 1
 2
 1
 2
 7
 6
 7
 3
 8
 2
 3
 3
 1
 6
 4
 5

3. Develop a new function called `crossvalidation` that receives:

- A matrix `targets::AbstractArray{Bool,2}`, where each row corresponds to one instance and each column to one class (multilabel format),
- An integer `k`, indicating the number of folds (subsets) to divide the dataset into.

The function must return a vector of integers of length `N` (where `N` is the number of rows in `targets`).  
Each element in the vector indicates the subset (fold) that the corresponding instance should be assigned to.

The partitioning must be **stratified**, meaning that the distribution of **each class across the folds is preserved**.
Suggested Steps:

    1. Create an index vector with as many elements as rows in `targets`.

    2. Use a loop that **iterates over the classes** (i.e., the columns of `targets`).  
     For each class:

       i. Count the number of instances that belong to that class using `sum(targets[:, i])`.  
       ii. Call the previously defined `crossvalidation` function with the number of instances and `k`.  
       iii. Assign the result to the corresponding positions in the index vector where `targets[:, i]` is `true`.

   ### Question 5.3
   > ❓ Could you combine these three operations into a single line inside the loop? (Optional exercise)

`Yes, the three operations can be combined into a single line inside the loop by directly assigning the output of the crossvalidation call to the positions corresponding to each class. For example:
folds[targets[:, i]] = crossvalidation(sum(targets[:, i]), k)
This performs the counting, function call, and assignment in one concise expression, without changing the overall logic or result.`

    3. Return the index vector.

  #### Important considerations

   - This function is allowed to use **a single loop** that iterates over the classes.
   - The result must be a vector of **integer values** with length equal to the number of instances `N`.

  #### Class support per fold

  It is essential to ensure that **each class has at least `k` instances**.

  > A typical value is `k = 10`.

### Question 5.4
> ❓ What would happen if a class has fewer than `k` instances?

`If a class has fewer than k instances, it becomes impossible to distribute at least one instance of that class into each fold. Some folds would end up without any samples from that class, breaking the stratification requirement. As a result, the model's evaluation might be biased or incomplete for that class, especially in imbalanced datasets.`

### Question 5.5
> ❓ How would this affect the calculation of metrics such as sensitivity or specificity?

`If some folds have no instances of a class, metrics like sensitivity or specificity cannot be properly calculated for that fold. Sensitivity (true positive rate) would be undefined because there are no positive cases to detect, and specificity (true negative rate) could be artificially inflated or meaningless. This leads to unreliable or biased evaluation of the model’s performance for that class.`

If, for any reason, it is not possible to guarantee at least `k` samples per class, you may consider **reducing the value of `k`**.  
In this case, consult your instructor to evaluate the implications and whether the trained models would still yield meaningful results.

In [145]:
function crossvalidation(targets::AbstractArray{Bool,2}, k::Int64)
    #TODO
    nInstances = size(targets,1)
    nClasses = size(targets,2)
    
    # 1. Create an index vector with as many elements as rows in `targets`.
    indices = Vector{Int64}(zeros(nInstances))

    # 2. Use a loop that **iterates over the classes** (i.e., the columns of `targets`).  
    for i in 1:nClasses
       # i. Count the number of instances that belong to that class using `sum(targets[:, i])`.  
       count = sum(targets[:, i])
       # ii. Call the previously defined `crossvalidation` function with the number of instances and `k`. 
       cv = crossvalidation(count, k) 
       # iii. Assign the result to the corresponding positions in the index vector where `targets[:, i]` is `true`.
       indices[targets[:, i]] = cv
    end

    # 3. Return the index vector.
    return indices
end

crossvalidation (generic function with 4 methods)

In [146]:
v2 = Array{Bool,2}(undef, 20, 3)
display(v2)
crossvalidation(v2, 6)

20×3 Matrix{Bool}:
 1  0  1
 0  0  0
 0  0  0
 0  0  0
 0  1  0
 0  0  0
 0  0  0
 0  0  0
 1  0  0
 0  0  0
 0  0  0
 0  0  0
 0  1  0
 0  0  0
 0  0  0
 0  0  0
 1  0  1
 0  0  0
 0  0  0
 0  0  0

20-element Vector{Int64}:
 2
 0
 0
 0
 1
 0
 0
 0
 2
 0
 0
 0
 2
 0
 0
 0
 1
 0
 0
 0

4. Implement a final version of the `crossvalidation` function, where:

- The first argument is `targets::AbstractArray{<:Any,1}`, a vector containing **heterogeneous elements** (e.g., strings, symbols, integers, etc.),
- The second argument is the usual integer `k` (number of folds),
- The output is a vector of **integer values**, one per instance, indicating the fold assignment.

To implement this function, follow these steps:

   1. Call the `oneHotEncoding` function, passing `targets` as the input.  
      This will convert the labels into a binary matrix (multilabel format).
   2. Call the **previous version** of the `crossvalidation` function, passing the encoded matrix and the value `k`.

   > **No loops are allowed** in this function.
   > The returned vector must contain integers and have the same length as the input.

In [147]:
# HELPER
function oneHotEncoding(feature::AbstractArray{<:Any,1},      
        classes::AbstractArray{<:Any,1})

    """
    Parameters
    ----------
    feature :: AbstractVector
        The input vector of categorical values to be encoded.
    classes :: AbstractVector
        The list/array of unique classes used as encoding reference.
    """
    """
    AbstractArray{<:Any,1} :
    AbstractArray → not restricted to just Vector, could be any 1-dimensional array type.
    1 → one-dimensional array (i.e. a vector).
    <:Any → element type can be any subtype of Any (which basically means no restriction at all).
    So effectively, AbstractArray{<:Any,1} is just a very general way of saying:
    “Accept any 1-D array, regardless of element type.”
    """
    # Defensive: ensure feature is a vector
    # Check that all feature values exist in the set of classes
    @assert(all([in(value, classes) for value in feature]))
    
    # Number of classes
    numClasses = length(classes)
    
    # Defensive: require at least two classes
    @assert(numClasses > 1)
    
    if (numClasses == 2)
        # Special case: binary classification, use a single column
        oneHot = reshape(feature .== classes[1], :, 1)
    else
        # General case: more than two classes
        oneHot = BitArray{2}(undef, length(feature), numClasses)
        for numClass = 1:numClasses
            # Mark 1 where feature matches the class
            oneHot[:, numClass] .= (feature .== classes[numClass])
        end
    end


    """
    Returns
    -------
    AbstractArray
        A one-hot encoded array:
        - Shape (n, 1) if there are 2 classes (binary case).
        - Shape (n, numClasses) if there are more than 2 classes.
    """
    return oneHot
end

oneHotEncoding (generic function with 1 method)

In [148]:
function crossvalidation(targets::AbstractArray{<:Any,1}, k::Int64)
    #TODO
    # 1. Call the `oneHotEncoding` function, passing `targets` as the input. This will convert the labels into a binary matrix (multilabel format).
    oneHotTargets = oneHotEncoding(targets, unique(targets))

    # 2. Call the **previous version** of the `crossvalidation` function, passing the encoded matrix and the value `k`.
    return crossvalidation(oneHotTargets, k)
end

crossvalidation (generic function with 4 methods)

In [149]:
targets = ["A", :B, 1, 2, "A", :B, 1, 2, "A", :B,
           1, 2, "A", :B, 1, 2, "A", :B, 1, 2,
           "A", :B, 1, 2, "A", :B, 1, 2, "A", :B,
           1, 2, "A", :B, 1, 2, "A", :B, 1, 2,
           "A", :B, 1, 2, "A", :B, 1, 2, "A", :B]

crossvalidation(targets, 3)

50-element Vector{Int64}:
 2
 3
 1
 1
 1
 3
 1
 1
 3
 1
 3
 1
 2
 ⋮
 2
 3
 1
 2
 3
 2
 2
 1
 2
 3
 3
 1

5. Finally, implement a function called `ANNCrossValidation` that trains an artificial neural network (ANN) multiple times following a **cross-validation strategy**, as described in this exercise.

 This function will receive both the usual inputs for training a neural network (as seen in previous exercises) and additional arguments for performing cross-validation with multiple executions per fold.

   #### **Function parameters**

   - `topology::AbstractArray{<:Int,1}` — Structure of the neural network.
   - `dataset::Tuple{AbstractArray{<:Real,2}, AbstractArray{<:Any,1}}` — A tuple with:
     - A matrix of inputs (instances in rows),
     - A vector of target labels (categorical). These must be converted into a Boolean matrix via `oneHotEncoding` inside this function.
   - `crossValidationIndices::Array{Int64,1}` — Vector of cross-validation fold assignments for each instance, obtained via the `crossvalidation` function.

   **Optional parameters** (with default values, as specified in the function signature):

   - `numExecutions`: Number of training repetitions per fold.
   - `transferFunctions`: Activation functions for each layer in the ANN.
     ### Question 5.6
     > ❓ Does it make sense to use a linear activation function in hidden layers?
    

`No, it generally does not make sense to use a linear activation function in hidden layers. Linear activations do not introduce non-linearity, so multiple linear layers collapse into a single linear transformation. This prevents the network from learning complex, non-linear patterns. Non-linear activation functions like sigmoid, ReLU, or tanh are preferred in hidden layers to allow the ANN to model complex relationships in the data.`

   - `maxEpochs`: Maximum number of training iterations (epochs).
   - `minLoss`: Minimum loss to stop training early.
   - `learningRate`: Learning rate.
   - `validationRatio`: Fraction of data used for validation in early stopping (can be 0).
   - `maxEpochsVal`: Maximum number of epochs without validation improvement before stopping early.


 **Step-by-step implementation**

   1. **Extract class labels**:
      ```julia
      classes = unique(targets)
      ```
   2. **One-hot encode** the categorical target labels using the `oneHotEncoding` function and the computed `classes`.
      ### Question 5.7
      > ❓ What could go wrong if the `classes` vector is not passed explicitly to `oneHotEncoding`?

`If the classes vector is not passed explicitly to oneHotEncoding, the function might determine the classes from the subset of target labels it sees in each fold. This can cause inconsistencies: some folds may miss certain classes, leading to one-hot vectors with different lengths or misaligned columns. As a result, the neural network could receive inconsistent target representations, causing errors in training or incorrect predictions.`

   3. **Determine the number of folds** using:
      ```julia
      numFolds = maximum(crossValidationIndices)
      ```
   4. Create one vector per metric (`accuracy`, `error rate`, `sensitivity`, `specificity`, `PPV`, `NPV`, `F1`) to store fold results.

   5. Initialize a **confusion matrix accumulator** (matrix of real numbers with all entries set to 0) for the **global test confusion matrix**.

   **For each fold**:

   - Extract `train` and `test` subsets for **inputs** and **outputs**, based on the cross-validation indices and current fold number.

   - Since ANNs are **non-deterministic**, results from a single training per fold may not be representative.  
     For this reason, train the ANN **multiple times per fold** (as specified in `numExecutions`).

   **Inside each fold**:

   1. Initialize vectors to store the metric results for each execution.  
   2. Create a 3D array of size `(numClasses, numClasses, numExecutions)` to store test confusion matrices from each execution.

   3. For each execution:

      - If `validationRatio > 0`, split the training set into training and validation sets using `holdOut`.  
        > ⚠️ You must adjust the ratio properly, since you’re applying it **on the training subset**, not the full dataset.
        
   ### Question 5.8  
   > ❓ How can you adapt the validation ratio accordingly?

`When you separate out a test set, the remaining data available for training and validation is smaller than the original dataset. If you naively apply the original validation fraction to this smaller pool, the number of validation samples would be too small relative to the intended proportion.
To correct this, we first computes how many validation samples the original fraction would imply for the full dataset. Then we compares this number to the size of the training-plus-validation subset and recalculates the validation fraction relative to that subset. This ensures that the validation set maintains the correct size proportionally, preserving the intended balance for early stopping or model evaluation.`

      - Train the network using `trainClassANN`.

      - Evaluate it on the test set using `confusionMatrix`.

      - Store the returned metrics and confusion matrix.

   4. After all executions for this fold:

      - Compute the **average** of each metric vector and store it in the global metric vectors.
      - Compute the **mean confusion matrix** using:
        ```julia
        mean(confusionMatrices, dims=3)
        ```
        > Note: This returns a 3D array with one slice; you must extract the 2D matrix from it.

      - Add the resulting matrix to the global confusion matrix.

#### Return values

Once all folds are completed:

- For each metric, compute the **mean and standard deviation** across the folds using `mean(...)` and `std(...)`.
- Return a tuple with 8 elements:
  1. Accuracy (mean, std)
  2. Error rate (mean, std)
  3. Sensitivity (mean, std)
  4. Specificity (mean, std)
  5. PPV (mean, std)
  6. NPV (mean, std)
  7. F1-score (mean, std)
  8. Global test confusion matrix

This function will be called in the next exercise by a general-purpose validation function.  
Unlike ANNs, **other ML models (e.g., SVM, kNN)** are **deterministic** — so they do not need the inner execution loop.  
For them, training once per fold is sufficient to produce stable results.

In [150]:
# HELPERS

function holdOut(N::Int, P::Real)
    #TODO
    @assert(0.0 < P < 1.0 , "P must be in (0,1)")
    #idx = shuffle(1:N)
    idx = randperm(N)
    #println("idx indices: ", idx)
    nTrain = Int(floor((1-P)*N))
    trainIdx = idx[1:nTrain]
    testIdx = idx[nTrain+1:end]
    return (trainIdx, testIdx)
end;
function holdOut(N::Int, Pval::Real, Ptest::Real)     
    @assert(0.0 < Pval < 1.0 , "Pval must be in (0,1)")
    @assert(0.0 < Ptest < 1.0 , "Ptest must be in (0,1)")
    @assert(Pval+Ptest < 1.0 , "Pval + Ptest must be in lower than 1")

    train_val_idx, test_idx = holdOut(N, Ptest)

    # Compute the absolute number of validation samples as Pval * N (rounded down)
    nVal = Int(floor(Pval*N))          
    # Compute the total number of samples available for training + validation (everything except the test set)
    nTrainVal = Int(floor((1-Ptest)*N)) 
    # Recalculate the validation proportion relative to the train+validation pool
    Pval2 = nVal/nTrainVal             

    local_train_idx, local_val_idx = holdOut(length(train_val_idx), Pval2)

    # Map from local indices back to the global indices
    train_idx = train_val_idx[local_train_idx]
    val_idx   = train_val_idx[local_val_idx]

    return (train_idx, val_idx, test_idx)
end;

function buildClassANN(numInputs::Int, topology::AbstractArray{<:Int,1}, numOutputs::Int;
                    transferFunctions::AbstractArray{<:Function,1}=fill(σ, length(topology))) 
    ann=Chain();
    numInputsLayer = numInputs;
    for numHiddenLayer in 1:length(topology)
        numNeurons = topology[numHiddenLayer];
        ann = Chain(ann..., Dense(numInputsLayer, numNeurons, transferFunctions[numHiddenLayer]));
        numInputsLayer = numNeurons;
    end;
    if (numOutputs == 1)
        ann = Chain(ann..., Dense(numInputsLayer, 1, σ));
    else
        ann = Chain(ann..., Dense(numInputsLayer, numOutputs, identity));
        ann = Chain(ann..., softmax);
    end;
    return ann;
end;   

function classifyOutputs(outputs::AbstractArray{<:Real,2}; 
                        threshold::Real=0.5) 
   numOutputs = size(outputs, 2);
    @assert(numOutputs!=2)
    if numOutputs==1
        return outputs.>=threshold;
    else
        # Look for the maximum value using the findmax funtion
        (_,indicesMaxEachInstance) = findmax(outputs, dims=2);
        # Set up then boolean matrix to everything false while max values aretrue.
        outputs = falses(size(outputs));
        outputs[indicesMaxEachInstance] .= true;
        # Defensive check if all patterns are in a single class
        @assert(all(sum(outputs, dims=2).==1));
        return outputs;
    end;
end;

function accuracy(outputs::Vector{Int64}, targets::Vector{Int64}) 
    mean(outputs.==targets);
end;

function trainClassANN(topology::AbstractArray{<:Int,1},  
            trainingDataset::Tuple{AbstractArray{<:Real,2}, AbstractArray{Bool,2}}; 
            # --- Requirement: optional validation dataset with default empty arrays ---
            validationDataset::Tuple{AbstractArray{<:Real,2}, AbstractArray{Bool,2}}= 
                    (Array{eltype(trainingDataset[1]),2}(undef,0,0), falses(0,0)), 
            # --- Requirement: optional test dataset with default empty arrays ---
            testDataset::Tuple{AbstractArray{<:Real,2}, AbstractArray{Bool,2}}= 
                    (Array{eltype(trainingDataset[1]),2}(undef,0,0), falses(0,0)), 
            transferFunctions::AbstractArray{<:Function,1}=fill(σ, length(topology)), 
            maxEpochs::Int=1000, minLoss::Real=0.0, learningRate::Real=0.01,  
            # --- Requirement: maxEpochsVal parameter (early stopping patience), default 20 ---
            maxEpochsVal::Int=20, showText::Bool=false) 

    # --- Unpacking datasets ---
    (inputs, targets) = trainingDataset
    (val_inputs, val_targets) = validationDataset
    (test_inputs, test_targets) = testDataset

    # --- Ensures dataset dimensions match ---
    @assert size(inputs,1) == size(targets,1)
    @assert size(val_inputs,1) == size(val_targets,1)
    @assert size(test_inputs,1) == size(test_targets,1)

    # --- Requirement: build ANN with given topology ---
    ann = buildClassANN(size(inputs,2), topology, size(targets,2))

    # --- Define loss function (binary or multi-class) ---
    # discriminates based on the number of output neurons
    loss(model,x,y) = (size(y,1) == 1) ? Losses.binarycrossentropy(model(x),y) : Losses.crossentropy(model(x),y)

    # --- Requirement: loss histories for training/validation/test ---
    trainingLosses = Float32[]
    validationLosses = Float32[]
    testLosses = Float32[]

    # --- Initial losses (cycle 0, before training) ---
    numEpoch = 0
    trainingLoss = loss(ann, inputs', targets')
    push!(trainingLosses, trainingLoss)
    
    # init message buffer
    log_message = []
    log_message = "Epoch $numEpoch - loss: $(round(trainingLoss, digits=4))"


    # --- if validation set is provided ---                          
    if size(val_inputs,1) > 0
        validationLoss = loss(ann, val_inputs', val_targets')
        push!(validationLosses, validationLoss)

        # update message buffer
        log_message *= " - val_loss: $(round(validationLoss, digits=4))"
    end

     # --- if test set is provided ---  
    if size(test_inputs,1) > 0
        testLoss = loss(ann, test_inputs', test_targets')
        push!(testLosses, testLoss)
        # update message buffer
        log_message *= " - test_loss: $(round(testLoss, digits=4))"
        if showText
            # do nothing
            #println("Epoch ", numEpoch, ": test loss: ", testLoss)
        end
    end

    if showText
        # print message buffer
        println(join(log_message))
    end
    # --- Optimizer setup ---
    opt_state = Flux.setup(Adam(learningRate), ann)

    # --- Requirement: variables for early stopping ---
    epochsWithoutImprovement = 0
    bestValLoss = Inf
    bestAnn = deepcopy(ann)  # Requirement: store best ANN (deepcopy to avoid overwriting)
    bestAnnEpoch = 0

    while (numEpoch < maxEpochs) && (trainingLoss > minLoss) && (epochsWithoutImprovement < maxEpochsVal)
        Flux.train!(loss, ann, [(inputs', targets')], opt_state)
        numEpoch += 1
        log_message = []
        # --- Compute training loss and store it ---
        trainingLoss = loss(ann, inputs', targets')
        push!(trainingLosses, trainingLoss)
        
        # update message buffer
        log_message = "Epoch $numEpoch - loss: $(round(trainingLoss, digits=4))"

        outputs=ann(inputs')
        outputs=classifyOutputs(outputs')
        predicted_classes = Flux.onecold(outputs')        # vector of predicted labels
        true_classes = Flux.onecold(targets')      # vector of true labels

        accuracy_train=accuracy(predicted_classes, true_classes)

        log_message *= " - acc: $(round(accuracy_train, digits=4))"

        # --- Requirement: if validation set provided, track its loss for early stopping ---
        if size(val_inputs,1) > 0
            validationLoss = loss(ann, val_inputs', val_targets')
            push!(validationLosses, validationLoss)

            # update message buffer
            log_message *= " - val_loss: $(round(validationLoss, digits=4))"

            outputs=ann(val_inputs')
            outputs=classifyOutputs(outputs')
            predicted_classes = Flux.onecold(outputs')        # vector of predicted labels
            true_classes = Flux.onecold(val_targets')      # vector of true labels

            accuracy_val=accuracy(predicted_classes, true_classes)

            log_message *= " - val_acc: $(round(accuracy_val, digits=4))"

            if validationLoss < bestValLoss
                bestValLoss = validationLoss
                epochsWithoutImprovement = 0
                bestAnn = deepcopy(ann)   # Requirement: update best ANN when improvement found
                bestAnnEpoch = numEpoch
            else
                epochsWithoutImprovement += 1
            end
        end

        # --- Requirement: also track test loss if provided ---
        if size(test_inputs,1) > 0
            testLoss = loss(ann, test_inputs', test_targets')
            push!(testLosses, testLoss)
            

            # update message buffer
            log_message *= " - test_loss: $(round(testLoss, digits=4))"

            outputs=ann(test_inputs')
            outputs=classifyOutputs(outputs')
            predicted_classes = Flux.onecold(outputs')        # vector of predicted labels
            true_classes = Flux.onecold(test_targets')      # vector of true labels

            accuracy_test=accuracy(predicted_classes, true_classes)
            test_error = 1-accuracy_test

            log_message *= " - test_acc: $(round(accuracy_test, digits=4))"
            log_message *= " - test_error: $(round(test_error, digits=4))"

        end
        
        if showText
            # update message buffer
            log_message *= " - epochsWithoutImprovement $(epochsWithoutImprovement)"
            println(join(log_message))
        end

        #print("trainingLoss > minLoss : $(trainingLoss > minLoss) \n")
        #print("epochsWithoutImprovement < maxEpochsVal : $(epochsWithoutImprovement < maxEpochsVal) \n")
    end  # closes while
    
    # --- Early stopping notice ---
    if (epochsWithoutImprovement >= maxEpochsVal) && showText
        println("⏹ Early stopping triggered after $numEpoch epochs (no improvement for $maxEpochsVal epochs).")
    end

    # --- Requirement: return the right ANN ---
    # If validation set was provided → return best ANN found
    # Otherwise → return last trained ANN
    finalAnn = size(val_inputs,1) > 0 ? bestAnn : ann

    bestEpoch = size(val_inputs,1) > 0 ? bestAnnEpoch : maxEpochs
    if showText
        println("The ANN corespond to the epoch $bestEpoch")
    end
    return finalAnn, trainingLosses, validationLosses, testLosses
end
function trainClassANN(topology::AbstractArray{<:Int,1},  
        trainingDataset::Tuple{AbstractArray{<:Real,2}, AbstractArray{Bool,1}}; 
        validationDataset::Tuple{AbstractArray{<:Real,2}, AbstractArray{Bool,1}}= 
                    (Array{eltype(trainingDataset[1]),2}(undef,0,0), falses(0)), 
        testDataset::Tuple{AbstractArray{<:Real,2}, AbstractArray{Bool,1}}= 
                    (Array{eltype(trainingDataset[1]),2}(undef,0,0), falses(0)), 
        transferFunctions::AbstractArray{<:Function,1}=fill(σ, length(topology)), 
        maxEpochs::Int=1000, minLoss::Real=0.0, learningRate::Real=0.01,  
        maxEpochsVal::Int=20, showText::Bool=false)

    (inputs, targets) = trainingDataset
    (val_inputs, val_targets) = validationDataset
    (test_inputs, test_targets) = testDataset
    
    # This function assumes that each sumple is in a row
    # we are going to check the numeber of samples to have same inputs and targets
    @assert(size(inputs,1)==size(targets,1));
    @assert (size(val_inputs,1) == size(val_targets,1));
    @assert (size(test_inputs,1) == size(test_targets,1));

    trainClassANN(topology, 
        (inputs, reshape(targets, length(targets), 1)),
        (val_inputs, reshape(val_targets, length(val_targets), 1)), 
        (test_inputs, reshape(test_targets, length(test_targets), 1)),
        transferFunctions, 
        maxEpochs=maxEpochs, minLoss=minLoss, learningRate=learningRate,
        maxEpochsVal, showText);
end;

function confusionMatrix(outrue_posuts::AbstractArray{Bool,1}, targets::AbstractArray{Bool,1})
    @assert length(outrue_posuts) == length(targets) "Outrue_posuts and targets must have the same length"
    
    # True positives, true negatives, false positives, false negatives
    true_pos = sum(outrue_posuts .& targets)
    true_neg = sum(.!outrue_posuts .& .!targets)
    false_pos = sum(outrue_posuts .& .!targets)
    false_neg = sum(.!outrue_posuts .& targets)

    # 1.Confusion matrix 
    cm = [true_pos false_pos; false_neg true_neg]

    # 2. Accuracy 
    accuracy = (true_pos + true_neg) / length(targets)

    # 3. Error rate
    error_rate = 1 - accuracy

    # 4. Recall
    sensitivity = if true_pos + false_neg == 0
        # All targets are negative
        1.0
    else
        true_pos / (true_pos + false_neg)
    end

    # 5. Specificity
    specificity = if true_neg + false_pos == 0
        # All targets are positive
        1.0
    else
        true_neg / (true_neg + false_pos)
    end

    # 6. Precision
    pos_pred_val = if true_pos + false_pos == 0
        if true_pos + false_neg == 0  # all patterns negative
            1.0
        else
            0.0
        end
    else
        true_pos / (true_pos + false_pos)
    end

    # 7. Negative predictive value
    neg_pred_val = if true_neg + false_neg == 0
        if true_neg + false_pos == 0  # all patterns positive
            1.0
        else
            0.0
        end
    else
        true_neg / (true_neg + false_neg)
    end

    # 8. F-score
    fscore = if sensitivity + pos_pred_val == 0
        0.0
    else
        2 * (pos_pred_val * sensitivity) / (pos_pred_val + sensitivity)
    end
    
    # returns results as a dictionary
    return (
        accuracy,
        error_rate,
        sensitivity,
        specificity,
        pos_pred_val,
        neg_pred_val,
        fscore,
        cm
    )
end
function confusionMatrix(outputs::AbstractArray{<:Real,1}, targets::AbstractArray{Bool,1}; threshold::Real=0.5)
    # Convert real-valued outputs to boolean predictions using the threshold
    bool_outputs = outputs .>= threshold
    
    # Call the boolean version
    return confusionMatrix(bool_outputs, targets)
end
function confusionMatrix(outputs::AbstractArray{Bool,2}, targets::AbstractArray{Bool,2}; weighted::Bool=true)
    #TODO
    
    # --- 1. Sanity checks ---
    n_classes_output = size(outputs, 2)
    n_classes_target = size(targets, 2)

    @assert n_classes_output == n_classes_target "Outputs and targets must have the same number of columns (classes)."

    # The first thing this function should do is to check that the number of columns of both matrices is equal and is different 
    # from 2. In case they have only one column,these columns are taken as vectors and the confusionMatrix function 
    # developed in the previous assignment is called.

    n_classes_output = size(outputs, 2)
    n_classes_target = size(targets, 2)
    @assert n_classes_output == n_classes_target "Outputs and targets must have the same number of columns (classes)."
    @assert n_classes_target >= 1 "Outputs and targets must have at least one column (class)."

    # --- 2. If single-column (binary case), call previous 1D confusionMatrix ---
    if n_classes_output == 1
        # Take columns as vectors
        y_pred = vec(outputs)
        y_true = vec(targets)
        return confusionMatrix(y_pred, y_true)  # call the binary version
    end
    
    # --- 3. Multi-class/multi-label
    n_classes = size(targets, 2)
    # Initialize vectors to store metrics per class
    sensitivities = zeros(Float64, n_classes)
    specificities = zeros(Float64, n_classes)
    positive_predicted_values = zeros(Float64, n_classes)
    negative_predicted_values = zeros(Float64, n_classes)
    f1_scores = zeros(Float64, n_classes)

    for class in 1:n_classes
        y_c_pred = outputs[:, class]
        y_c_true = targets[:, class]

        #Counts how many elements are true in both y_c_pred and y_c_true
        true_pos = sum(y_c_pred .& y_c_true) # "Apply the operation element by element across the two arrays."
                                             # & is the element-wise AND operator in Julia.

        # Counts how many samples are not predicted as class c and are not actually of class c
        true_neg = sum(.!y_c_pred .& .!y_c_true)

        # Counts how many samples are predicted as class c but are not actually of class c
        false_pos = sum(y_c_pred .& .!y_c_true)

        # Counts how many samples are not predicted as class c but are actually of class c
        false_neg = sum(.!y_c_pred .& y_c_true)
        
        sensitivities[class] = if true_pos + false_neg == 0
            # All targets are negative
            1.0
        else
            true_pos / (true_pos + false_neg)
        end

        specificities[class] = if true_neg + false_pos == 0
            # All targets are positive
            1.0
        else
            true_neg / (true_neg + false_pos)
        end
        
        #Precision = True Positives / (True Positives + False Positives
        positive_predicted_values[class] = if true_pos + false_pos == 0
            if true_pos + false_neg == 0
                # All targets are negative
                1.0
            else
                # No positive predictions, but there are positive targets
                0.0
            end
        else
            true_pos / (true_pos + false_pos)
        end


        #Negative Predictive Value = True Negatives / (True Negatives + False Negatives)
        negative_predicted_values[class] = if true_neg + false_neg == 0
            if true_neg + false_pos == 0
                # All targets are positive
                1.0
            else
                # No negative predictions, but there are negative targets
                0.0
            end
        else
            true_neg / (true_neg + false_neg)
        end

        #F1 Score = 2 * (Precision * Sensitivity) / (Precision + Sensitivity)
        f1_scores[class] = if sensitivities[class] + positive_predicted_values[class] == 0
            0.0
        else
            2 * (positive_predicted_values[class] * sensitivities[class]) / (positive_predicted_values[class] + sensitivities[class])
        end
    end 

    
    # Overall accuracy and error rate
    #accuracy = mean(outputs .== targets)
    # In this case (multi-class), we should count whether the whole row was predicted correctly, not each value.
    accuracy = mean([argmax(outputs[i, :]) == argmax(targets[i, :]) for i in 1:size(targets, 1)])
    error_rate = 1 - accuracy

    # Aggregate metrics
   # **Weighted**. In this stratey, the metrics corresponding to each class are averaged, weighting them with 
    # the number of patterns that belong (desired output) to each class. It is therefore suitable when classes are unbalanced.

    if weighted

        # When using the **weighted** strategy, it is necessary to compute how many instances belong 
        # to each class in order to calculate the weighted average.  
        class_counts = vec(sum(targets, dims=1))  # flatten row matrix into vector
        total_instances = sum(class_counts)
        weights = class_counts / total_instances
        sensitivity = sum(weights .* sensitivities)
        specificity = sum(weights .* specificities)
        positive_predicted_values = sum(weights .* positive_predicted_values)
        negative_predicted_values = sum(weights .* negative_predicted_values)
        fscore = sum(weights .* f1_scores)

    # - **Macro**. In this strategy, those metrics such as the PPV or the F-score are calculated as the arithmetic mean of the metrics of each class.
    #  As it is an arithmetic average, it does not consider the possible imbalance between classes.
    else
        sensitivity = mean(sensitivities)
        specificity = mean(specificities)
        positive_predicted_values = mean(positive_predicted_values)
        negative_predicted_values = mean(negative_predicted_values)
        fscore = mean(f1_scores)    
    end

    cm = [sum(targets[:, i] .& outputs[:, j]) for i in 1:n_classes, j in 1:n_classes]


    return (
        accuracy,
        error_rate,
        sensitivity,
        specificity,
        positive_predicted_values,
        negative_predicted_values,
        fscore,
        cm
    )
    
end
function confusionMatrix(outputs::AbstractArray{<:Real,2},targets::AbstractArray{Bool,2}; threshold::Real=0.5, weighted::Bool=true)
    #TODO

    # Convert continuous outputs into Boolean predictions
    classified_outputs = classifyOutputs(outputs, threshold=threshold)

    # Call the Boolean version of confusionMatrix
    return confusionMatrix(classified_outputs, targets; weighted=weighted)

end
function confusionMatrix(outputs::AbstractArray{<:Any,1}, targets::AbstractArray{<:Any,1}, classes::AbstractArray{<:Any,1}; weighted::Bool=true)
    # --- 1. Sanity checks ---
    @assert length(outputs) == length(targets) "Outputs and targets must have the same length."
    @assert (all([in(output, classes) for output in unique(outputs)])) "All output labels must exist in classes."
    @assert (all([in(target, classes) for target in unique(targets)])) "All target labels must exist in classes."

    # --- 2. One-hot encode both using the same classes vector ---
    outputs_onehot = oneHotEncoding(outputs, classes)
    targets_onehot = oneHotEncoding(targets, classes)

    # --- 3. Call the previous confusionMatrix version (the 2D boolean one) ---
    return confusionMatrix(outputs_onehot, targets_onehot; weighted=weighted)

end
function confusionMatrix(outputs::AbstractArray{<:Any,1}, targets::AbstractArray{<:Any,1}; weighted::Bool=true)
    #TODO
    # --- 1. Construct the classes vector from both targets and outputs ---
    classes = unique(vcat(targets, outputs))

    # --- 2. Call the previous version (that expects classes explicitly) ---
    return confusionMatrix(outputs, targets, classes; weighted=weighted)

    
end

confusionMatrix (generic function with 6 methods)

In [151]:
using Flux
using Flux.Losses
using Statistics

function ANNCrossValidation(topology::AbstractArray{<:Int,1},
        dataset::Tuple{AbstractArray{<:Real,2}, AbstractArray{<:Any,1}},
        crossValidationIndices::Array{Int64,1};
        numExecutions::Int=50,
        transferFunctions::AbstractArray{<:Function,1}=fill(σ, length(topology)),
        maxEpochs::Int=1000, minLoss::Real=0.0, learningRate::Real=0.01,
        validationRatio::Real=0, maxEpochsVal::Int=20)
    #TODO

    
    (inputs, targets) = dataset

    # 1. **Extract class labels**:
    classes = unique(targets)
    nClasses = size(classes,1)

    # 2. **One-hot encode** the categorical target labels using the `oneHotEncoding` function and the computed `classes`.
    oneHotTargets = oneHotEncoding(targets, classes)

    # 3. **Determine the number of folds** using:
    numFolds = maximum(crossValidationIndices)
    
    # 4. Create one vector per metric (`accuracy`, `error rate`, `sensitivity`, `specificity`, `PPV`, `NPV`, `F1`) to store fold results.
    accuracy = Float32[]
    error_rate = Float32[]
    sensitivity = Float32[]
    specificity = Float32[]
    ppv = Float32[]
    npv = Float32[]
    f1 = Float32[]

    # 5. Initialize a **confusion matrix accumulator** (matrix of real numbers with all entries set to 0) for the **global test confusion matrix**.
    cm_acc = zeros(Float32, nClasses, nClasses)

    # **For each fold**:
    for i in 1:numFolds
        # Extract `train` and `test` subsets for **inputs** and **outputs**, based on the cross-validation indices and current fold number.
        testInputs = inputs[crossValidationIndices .== i, :]
        testTargets = oneHotTargets[crossValidationIndices .== i, :]

        trainValInputs = inputs[crossValidationIndices .!= i, :]
        trainValTargets = oneHotTargets[crossValidationIndices .!= i, :]

        # Since ANNs are **non-deterministic**, results from a single training per fold may not be representative.  
        # For this reason, train the ANN **multiple times per fold** (as specified in `numExecutions`).
        # **Inside each fold**:
        # 1. Initialize vectors to store the metric results for each execution.  
        accuracy_fold = Float32[]
        error_rate_fold = Float32[]
        sensitivity_fold = Float32[]
        specificity_fold = Float32[]
        ppv_fold = Float32[]
        npv_fold = Float32[]
        f1_fold = Float32[]
        # 2. Create a 3D array of size `(numClasses, numClasses, numExecutions)` to store test confusion matrices from each execution.
        confusionMatrices = zeros(Float32, nClasses, nClasses, numExecutions)
        # 3. For each execution:
        for j in 1:numExecutions
            # If `validationRatio > 0`, split the training set into training and validation sets using `holdOut`.
            if validationRatio > 0
                # Recalculate the validation proportion relative to the train+validation pool
                N = size(inputs,1)
                nTrainVal = size(trainValInputs,1)
                nVal = Int(floor(validationRatio*N)) 
                realValidationRatio = nVal/nTrainVal

                (train_idx, val_idx) = holdOut(nTrainVal, realValidationRatio)

                trainInputs = trainValInputs[train_idx, :]
                valInputs = trainValInputs[val_idx, :]
                trainTargets = trainValTargets[train_idx, :]
                valTargets = trainValTargets[val_idx, :]
            end
            # Train the network using `trainClassANN`.
            finalANN, trainLoss, valLoss, testLoss = trainClassANN(
                topology,
                (trainInputs, trainTargets),
                validationDataset = (valInputs, valTargets),
                testDataset = (testInputs, testTargets)
            )

            # Evaluate it on the test set using `confusionMatrix`.
            testOutputs = finalANN(testInputs')
            testPredictions = classifyOutputs(testOutputs')
            testPredictionsClasses = Flux.onecold(testPredictions')
            testTargetClasses = Flux.onecold(testTargets')

            # Store the returned metrics and confusion matrix.
            accuracy_exec, error_rate_exec, sensitivity_exec, specificity_exec, ppv_exec, npv_exec, f1_exec, cm_exec = confusionMatrix(testPredictionsClasses, testTargetClasses)

            push!(accuracy_fold, accuracy_exec)
            push!(error_rate_fold, error_rate_exec)
            push!(sensitivity_fold, sensitivity_exec)
            push!(specificity_fold, specificity_exec)
            push!(ppv_fold, ppv_exec)
            push!(npv_fold, npv_exec)
            push!(f1_fold, f1_exec)
            confusionMatrices[:,:,j] = cm_exec
        end

        # 4. After all executions for this fold:
        # Compute the **average** of each metric vector and store it in the global metric vectors.
        push!(accuracy, mean(accuracy_fold))
        push!(error_rate, mean(error_rate_fold))
        push!(sensitivity, mean(sensitivity_fold))
        push!(specificity, mean(specificity_fold))
        push!(ppv, mean(ppv_fold))
        push!(npv, mean(npv_fold))
        push!(f1, mean(f1_fold))
        # Compute the **mean confusion matrix** using:
        cm_mean_fold = mean(confusionMatrices, dims=3)
        # This returns a 3D array with one slice; you must extract the 2D matrix from it.
        cm_mean_fold = dropdims(cm_mean_fold; dims=3)

        # Add the resulting matrix to the global confusion matrix.
        cm_acc += cm_mean_fold
    end
    return (
        (mean(accuracy), std(accuracy)),
        (mean(error_rate), std(error_rate)),
        (mean(sensitivity), std(sensitivity)),
        (mean(specificity), std(specificity)),
        (mean(ppv), std(ppv)),
        (mean(npv), std(npv)),
        (mean(f1), std(f1)),
        cm_acc / numFolds
    ) 
end

ANNCrossValidation (generic function with 1 method)

In [155]:
# Create a small example dataset
N = 1000   # number of instances
M = 3    # number of features
inputs = rand(N, M)
targets = [rand(["A", "B", "C"]) for _ in 1:N]


# Create cross-validation indices (e.g., 3 folds)
crossValidationIndices = crossvalidation(targets, 3)

# Call the ANNCrossValidation function
(accuracy_mean, accuracy_std), (error_rate_mean, error_rate_std), (sensitivity_mean, sensitivity_mean), (specifity_mean, specifity_std), (ppv_mean, ppv_std), (npv_mean, npv_std), (f1_mean, f1_std), cm = ANNCrossValidation([5, 4, 3], (inputs, targets), crossValidationIndices; numExecutions=3, maxEpochs=10, validationRatio=0.2)

println("Accuracy:")
println("\tmean: ", accuracy_mean)
println("\tstd: ", accuracy_std)
print("\n")
println("Error Rate:")
println("\tmean: ", error_rate_mean)
println("\tstd: ", error_rate_std)
print("\n")
println("Sensitivity:")
println("\tmean: ", sensitivity_mean)
println("\tstd: ", sensitivity_mean)
print("\n")
println("Specificity:")
println("\tmean: ", specifity_mean)
println("\tstd: ", specifity_std)
print("\n")
println("Positive Predictive Value (Precision):")
println("\tmean: ", ppv_mean)
println("\tstd: ", ppv_std)
print("\n")
println("Negative Predictive Value:")
println("\tmean: ", npv_mean)
println("\tstd: ", npv_std)
print("\n")
println("F-score:")
println("\tmean: ", f1_mean)
println("\tstd: ", f1_std)
print("\n")
println("Confusion Matrix:")
display(cm)

Accuracy:
	mean: 0.3349907
	std: 0.008612464

Error Rate:
	mean: 0.6650093
	std: 0.008612464

Sensitivity:
	mean: 0.008612464
	std: 0.008612464

Specificity:
	mean: 0.6601985
	std: 0.0057450593

Positive Predictive Value (Precision):
	mean: 0.12419736
	std: 0.01660422

Negative Predictive Value:
	mean: 0.46479273
	std: 0.038585294

F-score:
	mean: 0.17804062
	std: 0.012257536

Confusion Matrix:


3×3 Matrix{Float32}:
 33.3333  16.4444  61.2222
 33.3333  15.1111  60.8889
 33.7778  16.0     63.2222